In [8]:
from typing import Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import AnyMessage,AIMessage
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt,Command 
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage

In [9]:
load_dotenv()

True

In [ ]:
llm=ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langgraph.graph.meesage import add_messages

class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages]

In [ ]:
def chat_node(state:ChatState):
    decision=interrupt({
        "type":"approval",
        "reason":"Model is about to answer a user question.",
        "question":state["messages"][-1].content,
        "instruction":"Approve this question ? Yes /No"
    })

    if decision["approved"]=="no":
        return {"messages":[AIMessage(content="Not Approved")]}
    else:
        response=llm.invoke(state["messages"])
        return {"messages":[response]}

In [ ]:
# Build the graph : Start -> chat -> END 
builder=StateGraph(ChatState)
builder.add_edge("chat",chat_node)
builder.add_edge("chat",END)

# Checkpointer is required for interrupts
checkpointer=MemorySaver()

# Compile the app
app=builder.compile(checkpointer=checkpointer)

In [ ]:
# Create a new thread id for this converstion
config={"configurable":{"thread_id":'1234'}}

# Step 1 user ask a questions

initial_input={
    "messages":[
        {"user","Explain gradient descent in very simple terms"}
    ]
}

# Invoke the graph for the first time
result=app.invoke(initial_input,config=config)

In [ ]:
result 

In [ ]:
message=result['_interrupt__'][0].values

In [ ]:
user_input=input(f"\nBackend message -{message}\n Approve this question?(Y/N):")

In [ ]:
final_result=app.invoke(
    Command(resume={"approved":user_input}),
    config=config
)

In [ ]:
print(final_result)